# Relation K/V — D · pilot500 · T4×2
**Internet ON · T4×2 · Save & Run All. Mỗi notebook một arm, dùng cả2GPU.**
D: directional keys/values. E: giữ keys, chia sẻ value giữa hai equivalent views cùng fact.
Cả hai cùng C0 init, SCST+own-C0 KL+BCE+observed MP, endpoint500; chưa có kết quả mô hình.

Add Input: relation-kv-preparation-v115; concept-mass-c-c0-v111-v112;
prompt-cache; objectdetectionecache; mscoco-2014 của vuthetam.
Không cần raw visual-cache hoặc UPT dataset. Prompt-cache chỉ bổ sung đúng98nativeheldprompts
cho supporting decoder diagnostic; không sửa H5 nguồn hoặc tạo prompt từ nhãn.
Kiểm5path ở cell đầu. Dataset Private phải chia sẻ với tài khoản Ngân nếu chạy E ở đó.

FullSHA/preflight/20stepbenchmark phải qua trước training; forecast không vừa trần thì STOP,
không tự rút steps. GPUphase cap7200s, CPU scorer1800s; caps không ETA hoặc cách tính quota.
CPU bootstrap vẫn có thể tiêu quota khi accelerator bật. User báo khoảng20h/account.
Giữ checkpoint.bin và gửi review ZIP. Không tự promote/checkpoint selection/sweep/full run.


In [ ]:
from pathlib import Path
ARM='D'
PREPARED_ROOT=Path('/kaggle/input/datasets/ducanh2403/relation-kv-preparation-v115')
CHECKPOINT=Path('/kaggle/input/datasets/ducanh2403/concept-mass-c-c0-v111-v112/C0.bin')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
DETECTIONS=Path('/kaggle/input/datasets/ducanh2403/objectdetectionecache/objectdetectioncache.json')
LEGACY_PROMPT=Path('/kaggle/input/datasets/ducanh2403/prompt-cache/prompt_clip_tokens_cache.pt')
OUTPUT=Path('/kaggle/working/relation_kv_'+ARM+'_pilot')
REPORTED_GPU_HOURS_PER_ACCOUNT=20


In [ ]:
CORE='6685ae7abd21ad8e375a82536e5701ccb350ad47'
ASSETS={'experiments/semantic_grounding_method/kv_pilot_data.py': '8a7c747495729b5829b99a441c5c93e29d45c07563a598880e3183b469de1528', 'experiments/semantic_grounding_method/kv_pilot_model.py': '2279384a6947eefe875c96ea3fb6b996f3544548cd0e568b591197f109f585d0', 'experiments/semantic_grounding_method/kv_pilot_preflight.py': 'd674f49a66e6c9385c3db9adb9d7144642234e5ff142066e395e004351db6f8c', 'experiments/semantic_grounding_method/kv_pilot_training.py': '1dec55d8c6b53421006ee4a1d60ea0e2a590e1c72369a05356c40b1e3bd5e0d9', 'experiments/semantic_grounding_method/kv_pilot_evaluate.py': '05051ec6f28f591b4fa0022eb237bc0f71d4f5191fb91fec1e02023d89e059b1', 'experiments/semantic_grounding_method/run_kv_pilot.py': 'f0b3c8113f96ce56268b3549e038c08d8c5e19df0cc36caab48e346de1bfc8cf', 'experiments/semantic_grounding_method/test_kv_pilot_data.py': '9ab80f90e56e4416e94f0c6770559ae441f6ab3c1e5a41130a13e74680b99b2e', 'experiments/semantic_grounding_method/test_relation_kv_math.py': '0b3eca5419d20e6a544b1b22372b37d0e88e30a5d02e7f3737d7072f99cd2014', 'experiments/semantic_grounding_method/kv_data.py': '2d6fec1d00fea6dbc34396b60f868549ac86d17b6410f56ff47e3cf01ba28e39', 'experiments/semantic_grounding_method/KV_SAVE_V115_PACKAGE.json': '82521c3949d6129d024620aef6fe9e35f3d1074c4051d05a54a0db1ff712c964', 'experiments/semantic_grounding_method/KV_NATIVE_TEXT_AUDIT.json': '73fe5588e066522d0c236ffa78189ad091526e9c27b598cbc41ad464d7783506', 'experiments/semantic_grounding_method/data/gain_metadata.json.gz': '62f43f771aec9feba31a2fbbb6fe7db2878f259979faac378b74a754822c7b67', 'experiments/relation_frames_inventory/inventory.py': '8975c6a221e3d769c5449901347717f8776c4ae191fb4fad5d3eb117fe18e578', 'experiments/shared_pair_readout/metric_preflight.py': '6e29f54e57b0b2e7f6f62de57017061071444d254b09305411716713b86f83f4', 'captioning/__init__.py': 'ad259f50a32265d298d027a888f801d378c4669313b8dab06e0653978ce30633', 'captioning/models.py': '18026877836052c83e75c0d896adcc89822bb35616173c7e8b14b17c9a22e9d8', 'captioning/config.py': '4766d1bf82bf82b5199616c7d90ca74242f011e26f08db21b154b1c7d58e72bc', 'captioning/grounded_relations.py': '2c6b15048dce17a9bfa596f02af78866f0b7978f65d172414a3648c9aa49c58f', 'captioning/decoder_relation_memory.py': 'bc5086915bf8f35cf7161311785cf612b25695c5f0dbe03a760d8aca724dfc82', 'captioning/scst.py': '3a9a1f32fb27dae0447066e1388a31461f3f0781d52db5861880b037a572a78e', 'captioning/scst_runtime.py': '3452ccfbd43876dc6381b78d1229d1d6b05fe723e3863a04f31c4a784497f669', 'captioning/tokenizer.py': 'df2ace0025aa7d4e12c289ab7bca68046b48d4b6c2b5c12606a19ff5a50a638e', 'captioning/object_reward.py': '23282b1faca0b8e177ad8ce300f0b78ad4a5ab7b100e4b359b044025c567800c', 'captioning/object_consistency.py': '5fbd75b0ff4a35f0475d460a294894009d6007f468e248a0e4b528aaa8fabf73', 'captioning/semantic_prompts.py': 'a06f12dea63604f9db9d19b6922815db27a35b648a17059407ffaf26a2a0d511'}
import hashlib,json,os,signal,subprocess,sys,tempfile,threading,time,zipfile,queue,traceback,urllib.request
os.environ.update(CUBLAS_WORKSPACE_CONFIG=':4096:8',OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',
    PYTHONUNBUFFERED='1',PYTHONHASHSEED='42',TOKENIZERS_PARALLELISM='false',HDF5_USE_FILE_LOCKING='FALSE')
LOG=OUTPUT.with_name(OUTPUT.name+'_notebook.log');REVIEW=OUTPUT.with_name(OUTPUT.name+'_review.zip')
for p in (OUTPUT,LOG,REVIEW):
    if p.exists():raise FileExistsError('Preserve old output; use a fresh session/path: '+str(p))
for p in (CHECKPOINT,COCO_JSON,DETECTIONS,LEGACY_PROMPT,PREPARED_ROOT/'cpu_preparation.json',PREPARED_ROOT/'inputs.h5'):
    if not p.is_file():raise FileNotFoundError('Add Input or edit path: '+str(p))
WORK=Path(tempfile.mkdtemp(prefix='relation-kv-'+ARM+'-'))
REPO=WORK/'repo';REPO.mkdir()
HERE=REPO/'experiments/semantic_grounding_method/';CONFIG=WORK/'config.json'
print('Bootstrap starting; arm',ARM,'source',CORE,flush=True)

def run(args,timeout=900,cwd=None):
    args=list(map(str,args));print('Running:',' '.join(args),flush=True)
    process=subprocess.Popen(args,cwd=cwd or REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
        text=True,errors='replace',bufsize=1,start_new_session=True)
    messages=queue.Queue()
    def reader():
        try:
            for line in process.stdout:messages.put(line)
        finally:messages.put(None)
    thread=threading.Thread(target=reader,daemon=True);thread.start()
    start=heartbeat=time.monotonic();ended=False
    with LOG.open('a',encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line=messages.get(timeout=1)
                    if line is None:ended=True
                    else:print(line,end='',flush=True);log.write(line);log.flush()
                except queue.Empty:pass
                now=time.monotonic()
                if now-heartbeat>=30:
                    print('Worker active;',round(now-start),'seconds; see phase/step logs.',flush=True);heartbeat=now
                if now-start>timeout:raise TimeoutError('Locked phase cap exceeded; INCOMPLETE')
            if process.wait():raise subprocess.CalledProcessError(process.returncode,args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid,signal.SIGTERM)
                try:process.wait(timeout=10)
                except subprocess.TimeoutExpired:os.killpg(process.pid,signal.SIGKILL);process.wait()
            thread.join(timeout=5)


In [ ]:
try:
    for i,(relative,expected) in enumerate(ASSETS.items(),1):
        with urllib.request.urlopen('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+CORE+'/'+relative,timeout=60) as response:raw=response.read()
        if hashlib.sha256(raw).hexdigest()!=expected:raise ValueError('Changed pinned source '+relative)
        p=REPO/relative;p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(raw)
        if i%5==0 or i==len(ASSETS):print('Verified source files',i,'/',len(ASSETS),flush=True)
    os.environ['PYTHONPATH']=str(REPO)
    run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','pycocoevalcap==1.2','h5py','numpy','Pillow','nltk==3.9.2','future'])
    run([sys.executable,'-m','pip','install','-q','--no-deps','git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'])
    run([sys.executable,'-u','-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"])
    run(['java','-version'])
    CHAIR=WORK/'chair'
    run(['git','init',CHAIR]);run(['git','remote','add','origin','https://github.com/Maxlinn/CHAIR-metric-standalone.git'],cwd=CHAIR)
    run(['git','fetch','--depth','1','origin','4087a26211aa2339b9a76307cb8f0321ef691d0a'],cwd=CHAIR)
    run(['git','checkout','--detach','FETCH_HEAD'],cwd=CHAIR)
    sys.path.insert(0,str(REPO));sys.path.insert(0,str(HERE))
    from kv_pilot_data import POLICY
    cfg=dict(arm=ARM,prepared_root=str(PREPARED_ROOT),checkpoint=str(CHECKPOINT),dataset_json=str(COCO_JSON),
        detections=str(DETECTIONS),legacy_prompt=str(LEGACY_PROMPT),chair=str(CHAIR),output=str(OUTPUT),
        policy=POLICY,source_commit=CORE,source_assets=ASSETS,reported_gpu_hours_per_account=REPORTED_GPU_HOURS_PER_ACCOUNT)
    CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf-8')
except BaseException:
    with zipfile.ZipFile(REVIEW,'x',zipfile.ZIP_DEFLATED) as archive:
        if LOG.exists():archive.write(LOG,'notebook.log')
        archive.writestr('bootstrap_error.txt',traceback.format_exc())
    print('BOOTSTRAP FAILED; SEND',REVIEW,flush=True)
    raise


In [ ]:
success=False
try:
    run([sys.executable,'-u',HERE/'test_kv_pilot_data.py'],120)
    run([sys.executable,'-u',HERE/'test_relation_kv_math.py'],120)
    run([sys.executable,'-u',REPO/'experiments/shared_pair_readout/metric_preflight.py'],180)
    run([sys.executable,'-u',HERE/'run_kv_pilot.py','prepare','--config',CONFIG],1800)
    run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1',
        '--nproc_per_node=2','--max_restarts=0',HERE/'run_kv_pilot.py','worker','--config',CONFIG],7200)
    run([sys.executable,'-u',HERE/'run_kv_pilot.py','evaluate','--config',CONFIG],1800)
    result=json.loads((OUTPUT/'summary.json').read_bytes())
    if result['status']!='ENDPOINT_500_EVALUATED_WAIT_MATCHED_DE':raise RuntimeError('Incomplete endpoint')
    success=True
finally:
    with zipfile.ZipFile(REVIEW,'x',zipfile.ZIP_DEFLATED) as archive:
        if LOG.exists():archive.write(LOG,'notebook.log')
        if CONFIG.exists():archive.write(CONFIG,'notebook_config.json')
        for p in sorted(OUTPUT.glob('*')):
            if p.suffix in ('.json','.jsonl','.log'):archive.write(p,p.name)
        for relative in ASSETS:
            if relative.endswith('.py'):archive.write(REPO/relative,'source/'+relative)
        archive.writestr('execution_status.json',json.dumps(dict(success=success,arm=ARM,source_commit=CORE,assistant_launched_job=False)))
    print('SEND REVIEW ZIP:',REVIEW,flush=True)
    if (OUTPUT/'checkpoint.bin').exists():print('KEEP CHECKPOINT:',OUTPUT/'checkpoint.bin',flush=True)
    if not success:print('INCOMPLETE; do not compare a partial run with endpoint500.',flush=True)


Gửi `relation_kv_D_pilot_review.zip`. Giữ `checkpoint.bin`; không cần tải rawcache xuống máy.
Chờ đủ D/E để so cùng C0, nguồn/init/token identities và paired CIDEr/CHAIR/semantic scores.
Model/CUDA/gradient/parity chưa được Codex chạy; chỉ user-run preflight mới xác nhận.
